In [0]:
df = spark.table("main.aviation_departures.bronze_cle_departures")
display(df)

In [0]:
from pyspark.sql.functions import explode, col, to_timestamp, current_timestamp

flights_df = (
    df.select(explode("data").alias("flight"))
)

display(flights_df)

In [0]:
silver_df = (
    flights_df.select(
        col("flight.flight_date").alias("flight_date"),
        col("flight.flight_status").alias("flight_status"),

        col("flight.departure.airport").alias("dep_airport"),
        col("flight.departure.timezone").alias("dep_timezone"),
        col("flight.departure.iata").alias("dep_iata"),
        col("flight.departure.icao").alias("dep_icao"),
        col("flight.departure.terminal").alias("dep_terminal"),
        col("flight.departure.gate").alias("dep_gate"),
        col("flight.departure.delay").alias("dep_delay"),
        to_timestamp(col("flight.departure.scheduled")).alias("dep_scheduled_utc"),
        to_timestamp(col("flight.departure.estimated")).alias("dep_estimated_utc"),
        to_timestamp(col("flight.departure.actual")).alias("dep_actual_utc"),

        col("flight.arrival.airport").alias("arr_airport"),
        col("flight.arrival.timezone").alias("arr_timezone"),
        col("flight.arrival.iata").alias("arr_iata"),
        col("flight.arrival.icao").alias("arr_icao"),
        col("flight.arrival.terminal").alias("arr_terminal"),
        col("flight.arrival.gate").alias("arr_gate"),
        col("flight.arrival.delay").alias("arr_delay"),
        to_timestamp(col("flight.arrival.scheduled")).alias("arr_scheduled_utc"),
        to_timestamp(col("flight.arrival.estimated")).alias("arr_estimated_utc"),
        to_timestamp(col("flight.arrival.actual")).alias("arr_actual_utc"),

        col("flight.airline.name").alias("airline_name"),
        col("flight.airline.iata").alias("airline_iata"),
        col("flight.airline.icao").alias("airline_icao"),

        col("flight.flight.number").alias("flight_number"),
        col("flight.flight.iata").alias("flight_iata"),
        col("flight.flight.icao").alias("flight_icao"),

        col("flight.aircraft.registration").alias("aircraft_registration"),
        col("flight.aircraft.iata").alias("aircraft_iata"),
        col("flight.aircraft.icao").alias("aircraft_icao"),
        col("flight.aircraft.icao24").alias("aircraft_icao24"),

        current_timestamp().alias("silver_loaded_at")
    )
)

display(silver_df)

In [0]:
silver_df.write.format("delta").mode("overwrite").saveAsTable("main.aviation_departures.silver_cle_departures")